### Proxy Variables and Inference Risk in Lending Data

This notebook explores how seemingly non-sensitive variables can combine to reveal sensitive patterns in real-world data. Using the HMDA (Home Mortgage Disclosure Act) dataset, we demonstrate how machine learning models can leverage indirect signals—often called proxy variables—to make highly accurate predictions. We will demonstrate how multiple weak proxies combine into strong predictive signals and how proxies can encode sensitive patterns.

### Approach

This notebook follows these steps:

- Load and preprocess HMDA data
- Train a model to predict loan approval outcomes using only non-sensitive features
- Evaluate model performance
- Audit predictions across demographic groups (e.g., by sex)
- Analyze disparities to illustrate how proxy variables influence outcomes

In [40]:
import sys
sys.path.append(r"C:\Users\User\OneDrive\Documents\Personal_Projects\sgillihan.github.io\projects\proxy-data-research\code")
from pathlib import Path
from import_csv_from_zip import load_csv_from_zip
import pandas as pd
import numpy as np
import sklearn


ZIP_PATH = r"C:\Users\User\OneDrive\Documents\CSPB3112\Datasets\HMDA\hmda_2017_nationwide_all-records_labels.zip"

# Load a sample (chunked)
chunks = load_csv_from_zip(ZIP_PATH, chunksize=200_000)

df_list = []
max_chunks = 3   # ~600k rows total

for i, chunk in enumerate(chunks):
    df_list.append(chunk)
    if i >= max_chunks - 1:
        break

df_raw = pd.concat(df_list, ignore_index=True)

print(df_raw.shape)
display(df_raw.head())

No CSV filename specified. Loading first CSV found: hmda_2017_nationwide_all-records_labels.csv


C:\Users\User\AppData\Local\Temp\ipykernel_17232\2974510940.py:18: DtypeWarning: Columns (36,38,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(chunks):
C:\Users\User\AppData\Local\Temp\ipykernel_17232\2974510940.py:18: DtypeWarning: Columns (36,38,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(chunks):
C:\Users\User\AppData\Local\Temp\ipykernel_17232\2974510940.py:18: DtypeWarning: Columns (36,38,44,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(chunks):


(600000, 78)


,as_of_year,respondent_id,agency_name,agency_abbr,agency_code,loan_type_name,loan_type,property_type_name,property_type,loan_purpose_name,...,edit_status_name,edit_status,sequence_number,population,minority_population,hud_median_family_income,tract_to_msamd_income,number_of_owner_occupied_units,number_of_1_to_4_family_units,application_date_indicator
0,2017,75-2921540,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,NaN,3202.0,97.279999,57400.0,47.540001,710.0,1314.0,NaN
1,2017,0000504713,Consumer Financial Protection Bureau,CFPB,9,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,NaN,3733.0,4.580000,63900.0,86.239998,861.0,1241.0,NaN
2,2017,7810600004,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,NaN,5498.0,37.919998,75400.0,63.939999,1270.0,1658.0,NaN
3,2017,42-1739728,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,NaN,3566.0,11.830000,55200.0,74.290001,573.0,1261.0,NaN
4,2017,42-1739728,Department of Housing and Urban Development,HUD,7,Conventional,1,One-to-four family dwelling (other than manufa...,1,Refinancing,...,NaN,NaN,NaN,2910.0,48.660000,77500.0,79.250000,599.0,26.0,NaN


In [43]:
# Select relevant columns
cols = [
    "action_taken",
    "action_taken_name",
    "loan_amount_000s",
    "applicant_income_000s",
    "loan_purpose_name",
    "owner_occupancy_name",
    "property_type_name",
    "county_name",
    "county_code",
    "census_tract_number",
    "applicant_sex_name"
]

df_demo = df_raw[cols].copy()

print("Initial shape:", df_demo.shape)
display(df_demo.head())

# Define binary approval target
# 1 = originated
# 0 = otherwise (denied, withdrawn, etc.)
df_demo["approved"] = (df_demo["action_taken"] == 1).astype(int)

print(df_demo["approved"].value_counts(dropna=False))
print(df_demo["approved"].value_counts(normalize=True, dropna=False))

# Keep rows with usable values
df_demo = df_demo.dropna(subset=[
    "loan_amount_000s",
    "applicant_income_000s",
    "loan_purpose_name",
    "owner_occupancy_name",
    "property_type_name",
    "county_name",
    "census_tract_number",
    "applicant_sex_name"
]).copy()

# Keep only Male/Female for a simpler audit
df_demo = df_demo[df_demo["applicant_sex_name"].isin(["Male", "Female"])].copy()

# Remove zero or negative incomes to avoid ratio issues later
df_demo = df_demo[df_demo["applicant_income_000s"] > 0].copy()

print("Shape after cleaning:", df_demo.shape)
print(df_demo["applicant_sex_name"].value_counts())
print(df_demo["approved"].value_counts(normalize=True))

# Engineered features
df_demo["loan_income_ratio"] = (
    df_demo["loan_amount_000s"] / df_demo["applicant_income_000s"]
)

# Optional: simpler tract representation
df_demo["tract_str"] = df_demo["census_tract_number"].astype(str)

display(df_demo.head())

Initial shape: (600000, 11)


,action_taken,action_taken_name,loan_amount_000s,applicant_income_000s,loan_purpose_name,owner_occupancy_name,property_type_name,county_name,county_code,census_tract_number,applicant_sex_name
0,4,Application withdrawn by applicant,53.0,12.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Philadelphia County,101.0,173.00,Male
1,3,Application denied by financial institution,168.0,60.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Spokane County,63.0,127.01,Male
2,5,File closed for incompleteness,103.0,50.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Salt Lake County,35.0,1136.00,Male
3,1,Loan originated,88.0,53.0,Refinancing,Not owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Greene County,77.0,11.00,Female
4,4,Application withdrawn by applicant,90.0,29.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Cook County,31.0,306.04,Male


approved
0    323552
1    276448
Name: count, dtype: int64
approved
0    0.539253
1    0.460747
Name: proportion, dtype: float64
Shape after cleaning: (455083, 12)
applicant_sex_name
Male      305067
Female    150016
Name: count, dtype: int64
approved
0    0.500597
1    0.499403
Name: proportion, dtype: float64


,action_taken,action_taken_name,loan_amount_000s,applicant_income_000s,loan_purpose_name,owner_occupancy_name,property_type_name,county_name,county_code,census_tract_number,applicant_sex_name,approved,loan_income_ratio,tract_str
0,4,Application withdrawn by applicant,53.0,12.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Philadelphia County,101.0,173.00,Male,0,4.416667,173.0
1,3,Application denied by financial institution,168.0,60.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Spokane County,63.0,127.01,Male,0,2.800000,127.01
2,5,File closed for incompleteness,103.0,50.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Salt Lake County,35.0,1136.00,Male,0,2.060000,1136.0
3,1,Loan originated,88.0,53.0,Refinancing,Not owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Greene County,77.0,11.00,Female,1,1.660377,11.0
4,4,Application withdrawn by applicant,90.0,29.0,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Cook County,31.0,306.04,Male,0,3.103448,306.04


### Model 1: Weak Signals

The first model will demonstrate that financial parameters alone are only moderately predictive.

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

# Features (weak model)
features_1 = [
    "loan_amount_000s",
    "applicant_income_000s"
]

X = df_demo[features_1]
y = df_demo["approved"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Model 1: Financial features only
model_1 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

model_1.fit(X_train, y_train)

# Evaluate Model 1
y_pred = model_1.predict(X_test)
y_prob = model_1.predict_proba(X_test)[:, 1]

print("Model 1 (financial only)")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

Model 1 (financial only)
Accuracy: 0.517
ROC AUC: 0.541


The above accuracy for model 1 is barely above random 50/50. 

Below, we will introduce an engineered feature called loan_income_ratio

In [46]:
# Features (Model 2)
features_2 = [
    "loan_amount_000s",
    "applicant_income_000s",
    "loan_income_ratio"
]

X = df_demo[features_2]
y = df_demo["approved"]

# same split for fair comparison
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# model
model_2 = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

model_2.fit(X_train, y_train)

# evaluate
y_pred = model_2.predict(X_test)
y_prob = model_2.predict_proba(X_test)[:, 1]

print("Model 2 (financial + ratio)")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

Model 2 (financial + ratio)
Accuracy: 0.524
ROC AUC: 0.535


The accuracy level is still just above random. 

Below, we will add in additional parameters and train the model.

In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Feature split
numeric_features = [
    "loan_amount_000s",
    "applicant_income_000s",
    "loan_income_ratio"
]

categorical_features = [
    "loan_purpose_name",
    "owner_occupancy_name",
    "property_type_name",
    "county_name"
]

X = df_demo[numeric_features + categorical_features]
y = df_demo["approved"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Model 3: Financial + context + geography
model_3 = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])

model_3.fit(X_train, y_train)

# Evaluate
y_pred = model_3.predict(X_test)
y_prob = model_3.predict_proba(X_test)[:, 1]

print("Model 3 (financial + context + geography)")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

Model 3 (financial + context + geography)
Accuracy: 0.671
ROC AUC: 0.725


In [48]:
# Attach predictions to test set
df_test = X_test.copy()
df_test["approved_actual"] = y_test.values
df_test["pred"] = y_pred
df_test["pred_prob"] = y_prob

# bring gender back in
df_test["applicant_sex_name"] = df_demo.loc[X_test.index, "applicant_sex_name"]

display(df_test.head())

,loan_amount_000s,applicant_income_000s,loan_income_ratio,loan_purpose_name,owner_occupancy_name,property_type_name,county_name,approved_actual,pred,pred_prob,applicant_sex_name
494367,345.0,65.0,5.307692,Refinancing,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Jackson County,0,0,0.321938,Male
16107,236.0,63.0,3.746032,Home purchase,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Yellowstone County,1,1,0.699808,Female
82114,123.0,104.0,1.182692,Home purchase,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Summit County,1,1,0.708584,Male
28579,135.0,50.0,2.700000,Home purchase,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,Craighead County,0,1,0.644297,Male
126226,360.0,156.0,2.307692,Home purchase,Owner-occupied as a principal dwelling,One-to-four family dwelling (other than manufa...,San Diego County,1,1,0.780790,Male


In [49]:
df_test.groupby("applicant_sex_name")["pred"].mean()

applicant_sex_name
Female    0.496615
Male      0.540156
Name: pred, dtype: float64

In [50]:
df_test.groupby("applicant_sex_name")[["approved_actual", "pred"]].mean()

,approved_actual,pred
applicant_sex_name,,
Female,0.476117,0.496615
Male,0.510798,0.540156


In [51]:
from sklearn.metrics import confusion_matrix

def error_rates(group):
    cm = confusion_matrix(group["approved_actual"], group["pred"])
    tn, fp, fn, tp = cm.ravel()
    
    return pd.Series({
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp)
    })

df_test.groupby("applicant_sex_name").apply(error_rates)

C:\Users\User\AppData\Local\Temp\ipykernel_17232\396249381.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test.groupby("applicant_sex_name").apply(error_rates)


,FPR,FNR
applicant_sex_name,,
Female,0.321295,0.310477
Male,0.372726,0.299493
